# PROYECTO DEEP LEARNING

- Origen del dataset:
  
PlantVillage Dataset (disponible en Kaggle).
https://www.kaggle.com/datasets/arjuntejaswi/plant-village

- Tipo de datos:
Imágenes RGB (visión por computadora).
Imágenes formato JPG

- Estructura original:
Múltiples clases de enfermedades por tipo de planta.

Transformación para el proyecto:
Se agrupan todas las enfermedades en una sola clase:
Clase 0: Healthy
Clase 1: Diseased


- Modelo 1 — CNN desde cero

Capas Convolucionales + ReLU

MaxPooling

Dropout

Fully Connected

Capa final con Sigmoid

Modelo 2 — Transfer Learning

Modelo base: ResNet50 preentrenado en ImageNet

Congelamiento parcial de capas

Capa densa final con Sigmoid

Fine-tuning progresivo

Función de pérdida:
Binary Cross Entropy

Optimizador:
Adam

Regularización:

Dropout

Early stopping

Data augmentation


# CLASIFICACIÓN DE CARPETAS

- filepath (ruta completa a la imagen)
- original_class (nombre de carpeta)
- binary_label (0 si contiene “healthy”, 1 si no)
- binary_class (“healthy” o “diseased”
- Calcular conteos y proporciones (EDA de balance).
- Graficar barras

El dataset está organizado en carpetas. Si la carpeta dice “healthy”, esa imagen la consideramos sana; si no, enferma. Luego contamos cuántas hay de cada una para saber si el dataset está balanceado.


In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# 1) Ruta base donde están las carpetas (como en tu screenshot)
DATA_DIR = Path(r"C:\ITAM\MAESTRIA ITAM\SEMESTRE 2\APRENDIZAJE PROFUNDO\Proyecto\PlantVillage")

# 2) Extensiones comunes de imágenes
IMG_EXTS = {".jpg", ".jpeg", ".png"}

rows = []
for class_dir in DATA_DIR.iterdir():
    if not class_dir.is_dir():
        continue
    original_class = class_dir.name
    # Recorre imágenes dentro de cada carpeta
    for img_path in class_dir.rglob("*"):
        if img_path.suffix.lower() in IMG_EXTS:
            # Regla binaria: si la carpeta contiene "healthy" -> 0, si no -> 1
            is_healthy = "healthy" in original_class.lower()
            binary_label = 0 if is_healthy else 1
            binary_class = "healthy" if is_healthy else "diseased"
            rows.append({
                "filepath": str(img_path),
                "original_class": original_class,
                "binary_label": binary_label,
                "binary_class": binary_class
            })
df = pd.DataFrame(rows)
print("Total de imágenes:", len(df))
print("Total de clases originales (carpetas):", df["original_class"].nunique())
df.head()


# Balance de Clases 
Conteos y Proporciones 
- Las imágenes están guardadas en :  visualizaciones -> eda 


In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np

# Ruta
BASE_DIR = r"C:\ITAM\MAESTRIA ITAM\SEMESTRE 2\APRENDIZAJE PROFUNDO\Proyecto"
SAVE_DIR = os.path.join(BASE_DIR, "visualizaciones", "eda")
os.makedirs(SAVE_DIR, exist_ok=True)

# Datos
counts = df["binary_class"].value_counts().sort_index()
props = (counts / counts.sum())

# Colores oliva
colors = ["#6B8E23", "#556B2F"]

plt.figure(figsize=(8,6))
bars = plt.bar(counts.index, counts.values, color=colors)

# Títulos
plt.title("Balance de Clases (Clasificación Binaria)", fontsize=18, fontweight='bold')
plt.xlabel("Clase", fontsize=14, fontweight='bold')
plt.ylabel("Número de imágenes", fontsize=14, fontweight='bold')
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Ajuste dinámico para posición de texto
max_height = max(counts.values)

# Porcentajes arriba de cada barra
for i, bar in enumerate(bars):
    height = bar.get_height()
    percentage = props.iloc[i] * 100
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height + max_height * 0.02,   #  dinámico (2% arriba)
        f"{percentage:.1f}%",
        ha='center',
        fontsize=13,
        fontweight='bold'
    )

# Guardar	save_path = os.path.join(SAVE_DIR, "balance_clases_binario_estilizado.png")
plt.tight_layout()
plt.savefig(save_path, dpi=300)
plt.close()
print(f" Gráfica guardada en: {save_path}")


In [ ]:
# Carpetas que aportan más imágenes:
top_folders = df["original_class"].value_counts().head(15)
top_folders


RESULTADOS: 
TOTAL DE IMÁGENES : 20,637
- Diseased : 17416 (84.39%)
- Healthy : 3221 (15.61%)

ESTÁ SUPER DESBALANCEADO !!!!!!!
Por cada una 1 imagen healthy , hay 5.4 imagenes diseased 

################

Balance de clases

El dataset presenta un desbalance significativo entre clases.
El 84.39% de las imágenes corresponden a hojas enfermas, mientras que solo el 15.61% representan hojas saludables.

Esta asimetría implica que un modelo entrenado sin estrategias de corrección podría sesgarse hacia la clase mayoritaria, obteniendo métricas de accuracy aparentemente altas pero bajo desempeño en la detección de hojas sanas.

Por lo tanto, será necesario aplicar técnicas de balanceo o ponderación de clases durante el entrenamiento.
